In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import cv2, numpy as np
from sponsortrack import config
from sponsortrack.ingestion.logo_loader import load_logo

def guardar_png(ruta, img):  # imwrite que tolera acentos
    ok, buf = cv2.imencode(".png", img)
    buf.tofile(str(ruta))

sift = cv2.SIFT_create()
config.OUT_FIGURES.mkdir(parents=True, exist_ok=True)

for marca, info in config.BRANDS.items():
    for archivo in info["variantes"]:
        img, mask = load_logo(config.LOGOS_DIR / archivo, info["fondo_solo_borde"])
        w = config.SIFT_LOGO_WIDTH
        h = int(img.shape[0] * w / img.shape[1])
        g = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (w, h), interpolation=cv2.INTER_AREA)
        m = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
        kp = sift.detect(g, m)
        print(f"{archivo:22s} {img.shape[1]}x{img.shape[0]}  logo={100*(mask>0).mean():5.1f}%  keypoints@{w}px={len(kp)}")
        vis = img.copy(); vis[mask == 0] = (0, 255, 0)   # el fondo se pinta de verde
        guardar_png(config.OUT_FIGURES / f"mascara_{archivo}", vis)

fcb_clasico.png        530x538  logo= 74.0%  keypoints@160px=133
fcb_naranja.png        335x284  logo=100.0%  keypoints@160px=200
spotify_clasico.png    1737x1743  logo= 64.7%  keypoints@160px=1
spotify_alternativo.png 665x671  logo= 64.0%  keypoints@160px=4
nike_clasico.png       1238x444  logo= 27.0%  keypoints@160px=12
nike_alternativo.png   776x285  logo= 27.8%  keypoints@160px=4


In [2]:
video = [p for p in config.VIDEOS_DIR.iterdir() if p.is_file()][0]
mb = video.stat().st_size / 1e6
print(video.name, f"{mb:.0f} MB", f"≈ {mb*8/1034:.1f} Mbps reales")

cap = cv2.VideoCapture(str(video))
print("abre:", cap.isOpened(),
      "fps:", cap.get(cv2.CAP_PROP_FPS),
      "fotogramas:", cap.get(cv2.CAP_PROP_FRAME_COUNT),   # esperado ≈ 25 850
      "tamaño:", cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.set(cv2.CAP_PROP_POS_MSEC, 75_000)                     # segundo 1:15, como tu captura
ok, frame = cap.read()
print("lectura:", ok, frame.shape if ok else None)         # esperado (1080, 1920, 3)
if ok:
    guardar_png(config.OUT_FIGURES / "frame_1m15.png", frame)

VideoTarget.mp4 232 MB ≈ 1.8 Mbps reales
abre: True fps: 25.0 fotogramas: 25872.0 tamaño: 1920.0 1080.0
lectura: True (1080, 1920, 3)
